# Advanced Structures & Memory Optimization (__slots__): Beginner Guide

### 🌟 What Are Advanced Structures & Memory Optimization (`__slots__`)?
Python objects use dynamic dictionaries (`__dict__`) to store attributes by default, which introduces memory overhead. Using **`__slots__`**, along with specialized containers like `collections.deque` and `namedtuple`, significantly slashes memory usage when handling millions of objects.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Memory Inspection**: Covers `sys.getsizeof()` and `id()`.
- **Specialized Collections**: Covers `collections.defaultdict`, `Counter`, `deque`, `namedtuple`, and `OrderedDict`.
- **Low-Level Memory Optimization**: Covers `__slots__ = ('attr1', 'attr2')` eliminating dynamic `__dict__`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import json
import re
import collections
from datetime import datetime, timedelta

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get('transaction_amount'):
            transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 14635 transaction records from ../data/raw_transactions.csv


### 🔹 Memory Measurement: `sys.getsizeof()`
Measures shallow RAM consumption of Python objects in bytes. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `sys.getsizeof(obj)`


In [2]:
sample_dict = transactions[0]
print('Memory size of transaction dict:', sys.getsizeof(sample_dict), 'bytes')

Memory size of transaction dict: 464 bytes


### 🔹 Memory Address Pointer: `id()`
Returns unique memory address integer for the target object. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `id(obj)`


In [3]:
print('Memory address (id) of transaction dict:', id(sample_dict))

Memory address (id) of transaction dict: 2639233882752


### 🔹 Default Initialization: `collections.defaultdict`
Initializes missing dictionary keys automatically via factory function. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Use `dict.get(key, default)` when looking up keys that might not exist, avoiding unexpected `KeyError` crashes.

**Syntax:** `collections.defaultdict(list)`


In [4]:
card_totals = collections.defaultdict(float)
for t in transactions[:10]:
    card_totals[t['card_type']] += float(t['transaction_amount'] or 0.0)
print('defaultdict card totals:', dict(card_totals))

defaultdict card totals: {'Visa': 3523.47, 'MasterCard': 2169.61, 'Discover': 136.66, 'Amex': 905.86}


### 🔹 Element Tallies: `collections.Counter`
Counts frequency of hashable items in $O(N)$ linear time. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `collections.Counter(iterable)`


In [5]:
card_counts = collections.Counter([t['card_type'] for t in transactions])
print('Card frequencies (Counter):', card_counts)

Card frequencies (Counter): Counter({'Amex': 3713, 'MasterCard': 3670, 'Discover': 3628, 'Visa': 3624})


### 🔹 Double-Ended Queue: `collections.deque`
Provides $O(1)$ fast appends and pops from both left and right ends. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `collections.deque(maxlen=5)`


In [6]:
recent_queue = collections.deque(maxlen=3)
for t in transactions[:5]:
    recent_queue.append(t['transaction_id'])
print('Rolling deque (maxlen=3):', list(recent_queue))

Rolling deque (maxlen=3): ['TX108328', 'TX108563', 'TX107002']


### 🔹 Lightweight Named Tuples: `collections.namedtuple`
Creates lightweight tuple subclasses with dot-notation attribute access and zero `__dict__` overhead. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `collections.namedtuple('Tx', fields)`


In [7]:
TxRec = collections.namedtuple('TxRec', ['id', 'amount', 'card'])
nt_obj = TxRec(transactions[0]['transaction_id'], float(transactions[0]['transaction_amount']), transactions[0]['card_type'])
print('namedtuple fields:', nt_obj.id, nt_obj.amount, nt_obj.card)

namedtuple fields: TX110686 1216.33 Visa


### 🔹 Ordered Dictionaries: `collections.OrderedDict`
Dictionary remembering insertion order with re-ordering capabilities like `.move_to_end()`. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Use `dict.get(key, default)` when looking up keys that might not exist, avoiding unexpected `KeyError` crashes.

**Syntax:** `collections.OrderedDict()`


In [8]:
ord_dict = collections.OrderedDict()
ord_dict['first'] = 1
ord_dict['second'] = 2
ord_dict.move_to_end('first')
print('OrderedDict after move_to_end:', list(ord_dict.keys()))

OrderedDict after move_to_end: ['second', 'first']


### 🔹 Low-Level Memory Optimization: `__slots__`
Replaces dynamic per-instance `__dict__` with a fixed-size C-struct array, reducing instance memory by ~60%. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `__slots__ = ('id', 'amount')`


In [9]:
class RegularTx:
    def __init__(self, id, amt): self.id = id; self.amt = amt

class SlottedTx:
    __slots__ = ('id', 'amt')
    def __init__(self, id, amt): self.id = id; self.amt = amt

reg = RegularTx('TX1', 100.0)
slot = SlottedTx('TX1', 100.0)
print('Regular object size (with __dict__):', sys.getsizeof(reg) + sys.getsizeof(reg.__dict__), 'bytes')
print('Slotted object size (no __dict__):', sys.getsizeof(slot), 'bytes')

Regular object size (with __dict__): 344 bytes
Slotted object size (no __dict__): 48 bytes


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Memory Profiling on 10,000 Objects: Dict vs Slotted Class

**Approach:** Compare RAM consumption of storing 10,000 objects in memory.
**Syntax:** `[SlottedTx(...) for _ in range(N)]`


In [10]:
slotted_arr = [SlottedTx(t['transaction_id'], float(t['transaction_amount'] or 0.0)) for t in transactions[:5000]]
print(f'Memory for 5000 slotted objects: {sum(sys.getsizeof(o) for o in slotted_arr) / 1024:.1f} KB')

Memory for 5000 slotted objects: 234.4 KB
